# Lakehouses and Tables Deployer

This notebook automates the creation of Fabric Lakehouses and Delta tables from a structured directory of parquet files.

## Workflow Overview

1. **Create Lakehouses** - Create missing lakehouses in the workspace
2. **Load Manifest** - Load table allowlist from LakehouseHydrationManifest.json
3. **Discover Tables** - Scan source folders for table definitions
4. **Filter Tables** - Only include tables defined in the manifest for each lakehouse
5. **Create Delta Tables** - Load parquet data and create Delta tables in target lakehouses
6. **Report Results** - Display summary of created, skipped, and filtered tables
7. **Create Lakehouse Folders** - Create folder structures in lakehouses from manifest

## Manifest-Based Table Filtering

**🔒 Security Feature**: This notebook uses `LakehouseHydrationManifest.json` as the single source of truth for which tables to deploy.

- **Only tables listed in the manifest** for each lakehouse will be deployed
- Tables in the source folder but not in the manifest will be **filtered out** and logged
- This prevents accidental deployment of hundreds of unintended tables
- Ensures consistency between configuration and deployed resources

**Example:**
```
Manifest defines: bronze → ["ClinicalFhir", "ImagingDicom"]
Source has 100 tables, but only the 2 above will be deployed.
```

## Prerequisites

- Access to Microsoft Fabric workspace
- Source parquet files accessible at specified path
- LakehouseHydrationManifest.json deployed to dist folder
- Sufficient permissions to create lakehouses and tables

## Instructions

1. Update source configuration parameters in Cell 5:
   - `WORKSPACE_NAME` - Workspace container name
   - `SOURCE_LAKEHOUSE` - Source lakehouse name
   - `SOURCE_FOLDER_BASE` - Base path to healthcare tables
   - `HEALTHCARE_TABLES_VERSION` - Version to deploy
2. Review lakehouse mapping configuration
3. Verify LakehouseHydrationManifest.json contains correct table lists
4. Run Cell 12 to execute the complete deployment with manifest filtering

---

## Import Libraries

In [ ]:
%run common_deployment_config

In [ ]:
print("✓ Using configuration from common_deployment_config")

---

## Configuration Parameters

Configure the source data location and target lakehouse mappings below.

### Source Configuration
- `WORKSPACE_NAME` - Workspace container name where source data resides
- `SOURCE_LAKEHOUSE` - Lakehouse name containing source parquet files
- `SOURCE_FOLDER_BASE` - Base path to healthcare tables folder
- `HEALTHCARE_TABLES_VERSION` - Version of healthcare tables to deploy

### Target Configuration
- `FOLDER_TO_LAKEHOUSE_MAP` - Maps source folder names to target lakehouse names
- `LAKEHOUSES_TO_CREATE` - List of lakehouses to create if missing

> **Note:** OneLake endpoint is auto-detected from the Fabric environment.

In [ ]:
# ============================================================================
# NOTEBOOK-SPECIFIC CONFIGURATION  
# ============================================================================
# (Common config loaded from common_deployment_config)

# Construct tables-specific path from base
TABLES_SOURCE_PATH = f"{BASE_DIST_PATH}/healthcare-tables/{ARTIFACT_VERSION}"

print("✓ Tables deployer configuration:")
print(f"  Workspace: {WORKSPACE_NAME}")
print(f"  Version: {ARTIFACT_VERSION}")
print(f"  Tables Source: {TABLES_SOURCE_PATH}")

---

## Define Functions

All functions are defined below for lakehouse creation and table deployment.

In [ ]:
def create_lakehouses(lakehouses_to_create: List[str], workspace_id: str) -> Tuple[List[str], List[str]]:
    """
    Create lakehouses in the workspace if they don't already exist.
    
    Args:
        lakehouses_to_create: List of lakehouse names to create
        workspace_id: Fabric workspace ID
        
    Returns:
        Tuple of (created_lakehouses, existing_lakehouses)
    """
    print("=" * 80)
    print("CREATING LAKEHOUSES")
    print("=" * 80)
    
    client = FabricRestClient()
    created = []
    already_exists = []
    
    # Get all existing Lakehouses in the workspace
    try:
        df_items = fabric.list_items(workspace=workspace_id)
        existing_lakehouses = set(df_items[df_items['Type'] == 'Lakehouse']['Display Name'])
    except Exception as e:
        print(f"⚠️  Failed to retrieve Lakehouse list: {e}")
        existing_lakehouses = set()
    
    print(f"\nℹ️  Found {len(existing_lakehouses)} existing lakehouses in workspace")
    print(f"ℹ️  Processing {len(lakehouses_to_create)} lakehouses...\n")
    
    # Create only the missing Lakehouses
    for lh in lakehouses_to_create:
        # Apply prefixes to lakehouse name to match table creation logic
        prefixed_lh_name = build_artifact_name(lh)
        
        if prefixed_lh_name in existing_lakehouses:
            print(f"  ✓ Already exists: {prefixed_lh_name}")
            already_exists.append(prefixed_lh_name)
            continue
        
        try:
            lakehouse_id = fabric.create_lakehouse(
                display_name=prefixed_lh_name,
                description=f"{prefixed_lh_name} lakehouse",
                workspace=workspace_id
            )
            print(f"  ✓ Created: {prefixed_lh_name} (ID: {lakehouse_id})")
            created.append(prefixed_lh_name)
        except Exception as e:
            print(f"  ✗ Failed to create {prefixed_lh_name}: {str(e)}")
    
    print(f"\n✓ Lakehouse creation complete")
    print(f"  Created: {len(created)}")
    print(f"  Already existed: {len(already_exists)}")
    print("=" * 80)
    
    return created, already_exists

print("✓ create_lakehouses() defined")

In [ ]:
def _is_delta_table_source(source_path: str) -> bool:
    """
    Check if a source table folder contains a _delta_log (i.e., is a full Delta table).
    
    Args:
        source_path: Path to the table folder in the artifact store
        
    Returns:
        True if _delta_log exists, False otherwise
    """
    try:
        delta_log_path = f"{source_path}/_delta_log"
        return mssparkutils.fs.exists(delta_log_path)
    except Exception:
        return False


def _validate_delta_copy(target_path: str, table_name: str) -> bool:
    """
    Validate that a Delta table was copied correctly by checking:
    1. _delta_log exists at target
    2. At least one JSON commit file exists in _delta_log
    
    Args:
        target_path: OneLake path where table was copied
        table_name: Name of the table (for logging)
        
    Returns:
        True if validation passes, False otherwise
    """
    try:
        delta_log_path = f"{target_path}/_delta_log"
        if not mssparkutils.fs.exists(delta_log_path):
            print(f"      ❌ Validation failed: _delta_log missing at {target_path}")
            return False
        
        # Check that at least one commit file exists
        log_files = mssparkutils.fs.ls(delta_log_path)
        json_files = [f for f in log_files if f.name.endswith(".json")]
        if not json_files:
            print(f"      ❌ Validation failed: No JSON commit files in _delta_log for {table_name}")
            return False
        
        return True
    except Exception as e:
        print(f"      ❌ Validation error for {table_name}: {e}")
        return False


def _wait_for_table_registration(full_table_name: str, max_retries: int = 5, wait_seconds: int = 3) -> bool:
    """
    Wait for Fabric to auto-register a Delta table after file copy.
    
    Args:
        full_table_name: Fully qualified table name (lakehouse.table)
        max_retries: Maximum number of retries
        wait_seconds: Seconds to wait between retries
        
    Returns:
        True if table is registered, False if not after all retries
    """
    import time
    
    for attempt in range(max_retries):
        try:
            if spark.catalog.tableExists(full_table_name):
                return True
        except Exception:
            pass
        
        if attempt < max_retries - 1:
            time.sleep(wait_seconds)
    
    return False


def discover_and_create_tables(source_base_path: str, folder_to_lakehouse_map: Dict[str, str], use_manifest_filter: bool = True) -> Tuple[List[str], List[str], List[str]]:
    """
    Discover tables in source folders and deploy Delta tables to target lakehouses.
    
    Deployment strategy:
    - If source folder contains _delta_log: copies the entire Delta table structure
      (preserving schema, column names, and table properties exactly as built).
    - If source folder contains only parquet files: falls back to read + saveAsTable.
    
    Tables are filtered based on LakehouseHydrationManifest.json if use_manifest_filter=True.
    
    Args:
        source_base_path: Base path containing layer folders with table data
        folder_to_lakehouse_map: Mapping of folder names to target lakehouse names
        use_manifest_filter: If True, only deploy tables listed in manifest (default: True)
        
    Returns:
        Tuple of (created_tables, skipped_tables, filtered_tables)
    """
    import time

    print("=" * 80)
    print("DISCOVERING AND CREATING TABLES")
    print("=" * 80)
    
    created_tables = []
    skipped_tables = []
    filtered_tables = []
    
    # Get all layer folders
    layer_folders = [f for f in mssparkutils.fs.ls(source_base_path) if f.isDir]
    print(f"\nℹ️  Found {len(layer_folders)} layer folders")
    print(f"ℹ️  Manifest filtering: {'ENABLED' if use_manifest_filter else 'DISABLED'}\n")
    
    for layer_folder in layer_folders:
        layer_name = layer_folder.name.rstrip("/")
        
        # Normalize to lowercase for case-insensitive lookup
        # (Manifest uses lowercase, but disk folders may be capitalized)
        layer_name_normalized = layer_name.lower()
        
        # Check if layer has a mapping
        if layer_name_normalized not in folder_to_lakehouse_map:
            print(f"⚠️  No mapping for '{layer_name}'. Skipping.\n")
            continue
        
        # Get lakehouse name(s) for this folder - may be multiple (e.g., poa_gold + cma_gold for Gold)
        lakehouse_names = folder_to_lakehouse_map[layer_name_normalized]
        if not isinstance(lakehouse_names, list):
            lakehouse_names = [lakehouse_names]  # Backward compatibility: convert single value to list
        
        layer_path = f"{source_base_path}/{layer_name}"
        
        # Process tables for EACH lakehouse mapped to this folder
        for base_lakehouse_name in lakehouse_names:
            # Apply prefix pattern to lakehouse name (but NOT to table names)
            target_lakehouse = build_artifact_name(base_lakehouse_name)
            
            print(f"{'='*60}")
            print(f"📁 Layer: {layer_name} → Lakehouse: {target_lakehouse}")
            print(f"{'='*60}")
            
            # Get allowed tables from manifest if filtering is enabled
            allowed_tables = None
            if use_manifest_filter:
                # Get manifest key for this lakehouse
                manifest_key = get_manifest_lakehouse_key(base_lakehouse_name)
                if manifest_key:
                    allowed_tables = get_lakehouse_tables(manifest_key)
                    print(f"   📋 Manifest defines {len(allowed_tables)} allowed table(s) for {manifest_key}")
                else:
                    print(f"   ⚠️  No manifest entry found for '{base_lakehouse_name}' - deploying all tables")
            
            # Get all table folders in this layer
            discovered_table_folders = [f for f in mssparkutils.fs.ls(layer_path) if f.isDir]
            discovered_count = len(discovered_table_folders)
            
            # Filter tables based on manifest if enabled
            if use_manifest_filter and allowed_tables is not None:
                # Convert to set for efficient lookup
                allowed_tables_set = set(allowed_tables)
                
                # Separate allowed and filtered tables
                table_folders = []
                for table_folder in discovered_table_folders:
                    table_name = table_folder.name.rstrip("/")
                    if table_name in allowed_tables_set:
                        table_folders.append(table_folder)
                    else:
                        filtered_tables.append(f"{target_lakehouse}.{table_name}")
                
                filtered_count = discovered_count - len(table_folders)
                print(f"   🔍 Discovered {discovered_count} table(s), filtered {filtered_count}, deploying {len(table_folders)}\n")
                
                # Log filtered tables if any
                if filtered_count > 0:
                    print(f"   ⏭️  Filtered tables (not in manifest):")
                    for table_folder in discovered_table_folders:
                        table_name = table_folder.name.rstrip("/")
                        if table_name not in allowed_tables_set:
                            print(f"      • {table_name}")
                    print()
            else:
                table_folders = discovered_table_folders
                print(f"   🔍 Discovered {discovered_count} table(s), deploying all\n")
            
            # Build OneLake target base path for this lakehouse
            lakehouse_tables_base = f"abfss://{WORKSPACE_NAME}@{ENDPOINT_URI}/{target_lakehouse}.Lakehouse/Tables"
            
            # Deploy tables
            for table_folder in table_folders:
                table_name = table_folder.name.rstrip("/")
                full_table_name = f"{target_lakehouse}.{table_name}"
                source_table_path = f"{layer_path}/{table_name}"
                target_table_path = f"{lakehouse_tables_base}/{table_name}"
                
                try:
                    # Check if table already exists
                    if spark.catalog.tableExists(full_table_name):
                        print(f"    ⚠️  Skipped: {full_table_name} (already exists)")
                        skipped_tables.append(full_table_name)
                        continue
                    
                    # Determine deployment strategy based on source structure
                    if _is_delta_table_source(source_table_path):
                        # Strategy: Direct Delta copy (preserves schema exactly)
                        print(f"    📋 Deploying (Delta copy): {full_table_name}")
                        
                        # Copy entire Delta table folder to lakehouse Tables/ path
                        mssparkutils.fs.cp(source_table_path, target_table_path, recurse=True)
                        
                        # Validate the copy
                        if not _validate_delta_copy(target_table_path, table_name):
                            print(f"    ✗ Copy validation failed for {full_table_name}, cleaning up...")
                            try:
                                mssparkutils.fs.rm(target_table_path, recurse=True)
                            except Exception:
                                pass
                            continue
                        
                        # Wait for Fabric to auto-register the table
                        if _wait_for_table_registration(full_table_name):
                            print(f"    ✓ Created (Delta copy): {full_table_name}")
                        else:
                            # Fallback: explicit registration
                            print(f"      ℹ️  Auto-registration pending, registering explicitly...")
                            spark.sql(f"CREATE TABLE IF NOT EXISTS `{target_lakehouse}`.`{table_name}` USING DELTA LOCATION '{target_table_path}'")
                            print(f"    ✓ Created (Delta copy + registered): {full_table_name}")
                        
                        created_tables.append(full_table_name)
                    
                    else:
                        # Fallback strategy: Read parquet + saveAsTable (for sources without _delta_log)
                        print(f"    📋 Deploying (parquet fallback): {full_table_name}")
                        parquet_path = f"{source_table_path}/*.parquet"
                        df = spark.read.format("parquet").load(parquet_path)
                        df.write.format("delta") \
                            .mode("overwrite") \
                            .option("overwriteSchema", "true") \
                            .saveAsTable(full_table_name)
                        
                        print(f"    ✓ Created (parquet fallback): {full_table_name}")
                        created_tables.append(full_table_name)
                    
                except Exception as e:
                    print(f"    ✗ Error deploying {full_table_name}: {e}")
            
            print()  # Blank line between lakehouses
    
    print("✓ Table discovery and creation complete")
    print(f"  Created:  {len(created_tables)} tables")
    print(f"  Skipped:  {len(skipped_tables)} tables (already existed)")
    print(f"  Filtered: {len(filtered_tables)} tables (not in manifest)")
    print("=" * 80)
    
    return created_tables, skipped_tables, filtered_tables

print("✓ discover_and_create_tables() defined")


In [ ]:
def flatten_folder_paths(folders: List[Dict[str, Any]], parent_path: str = "") -> List[str]:
    """
    Recursively flatten nested folder structure from manifest into leaf paths.
    
    Only leaf folders (those without subFolders) are returned, since
    mssparkutils.fs.mkdirs() creates all intermediate directories automatically.
    
    Args:
        folders: List of folder dicts from manifest (each has 'name' and optional 'subFolders')
        parent_path: Accumulated parent path for recursion
        
    Returns:
        Sorted, deduplicated list of leaf folder paths
        
    Examples:
        >>> flatten_folder_paths([{"name": "A", "subFolders": [{"name": "B"}]}])
        ['A/B']
        >>> flatten_folder_paths([{"name": "A"}, {"name": "C"}])
        ['A', 'C']
    """
    paths = []
    for folder in folders:
        name = folder.get("name", "")
        
        # Validate folder name
        if not name or not name.strip():
            print(f"  ⚠️  Skipping folder with empty name")
            continue
        if ".." in name:
            print(f"  ⚠️  Skipping folder with unsafe name: {repr(name)}")
            continue
        if "\\" in name:
            print(f"  ⚠️  Skipping folder with backslash in name: {repr(name)}")
            continue
        
        current_path = f"{parent_path}/{name}" if parent_path else name
        sub_folders = folder.get("subFolders", [])
        
        if sub_folders:
            paths.extend(flatten_folder_paths(sub_folders, current_path))
        else:
            paths.append(current_path)
    
    return sorted(set(paths))

print("✓ flatten_folder_paths() defined")


In [ ]:
def create_lakehouse_folders(
    lakehouses_to_create: List[str],
    workspace_id: str,
    endpoint_uri: str
) -> Tuple[int, int, int]:
    """
    Create folder structures in lakehouses based on the manifest.
    
    For each lakehouse that has folders defined in LakehouseHydrationManifest.json,
    creates the leaf folder paths in OneLake using mssparkutils.fs.mkdirs().
    Idempotent: skips folders that already exist.
    
    Args:
        lakehouses_to_create: List of base lakehouse names (from LAKEHOUSES_TO_CREATE)
        workspace_id: Fabric workspace ID
        endpoint_uri: OneLake endpoint URI
        
    Returns:
        Tuple of (created_count, skipped_count, error_count)
        
    Raises:
        RuntimeError: If workspace items cannot be listed, or if ALL folder
                      creation attempts fail (partial failures are logged but not raised)
    """
    print("=" * 80)
    print("CREATING LAKEHOUSE FOLDERS")
    print("=" * 80)
    
    total_created = 0
    total_skipped = 0
    total_errors = 0
    lakehouses_processed = 0
    
    # Get all existing lakehouses in workspace (single API call)
    try:
        df_items = fabric.list_items(workspace=workspace_id)
        lakehouse_items = df_items[df_items["Type"] == "Lakehouse"]
    except Exception as e:
        print(f"\n❌ Failed to retrieve lakehouse list: {e}")
        print("   Cannot proceed with folder creation without lakehouse information.")
        raise RuntimeError(f"Failed to list workspace items: {e}") from e
    
    print(f"\nℹ️  Found {len(lakehouse_items)} lakehouses in workspace")
    
    for base_name in lakehouses_to_create:
        # Get folders from manifest for this lakehouse
        try:
            manifest_folders = get_lakehouse_folders(base_name)
        except KeyError:
            continue
        
        if not manifest_folders:
            continue
        
        # Resolve lakehouse display name and ID
        prefixed_name = build_artifact_name(base_name)
        matching = lakehouse_items[lakehouse_items["Display Name"] == prefixed_name]
        
        if matching.empty:
            print(f"\n⚠️  Lakehouse '{prefixed_name}' not found in workspace — skipping folder creation")
            continue
        
        lakehouse_id = matching.iloc[0]["Id"]
        lakehouses_processed += 1
        
        # Flatten folder structure to leaf paths
        leaf_paths = flatten_folder_paths(manifest_folders)
        
        if not leaf_paths:
            print(f"\n⚠️  No valid folder paths found for '{prefixed_name}' — skipping")
            continue
        
        onelake_base = f"abfss://{WORKSPACE_NAME}@{endpoint_uri}/{prefixed_name}.Lakehouse/Files"
        
        print(f"\n{'='*60}")
        print(f"📁 {prefixed_name} — {len(leaf_paths)} folders to process")
        print(f"{'='*60}")
        
        lh_created = 0
        lh_skipped = 0
        lh_errors = 0
        
        for folder_path in leaf_paths:
            full_path = f"{onelake_base}/{folder_path}"
            try:
                if mssparkutils.fs.exists(full_path):
                    lh_skipped += 1
                    continue
                
                mssparkutils.fs.mkdirs(full_path)
                lh_created += 1
                print(f"  ✓ Created: {folder_path}")
                
            except Exception as e:
                lh_errors += 1
                print(f"  ✗ Failed: {folder_path} — {e}")
        
        print(f"\n  Summary for {prefixed_name}:")
        print(f"    Created: {lh_created} | Already existed: {lh_skipped} | Errors: {lh_errors}")
        
        total_created += lh_created
        total_skipped += lh_skipped
        total_errors += lh_errors
    
    print(f"\n✓ Folder creation complete")
    print(f"  Lakehouses processed: {lakehouses_processed}")
    print(f"  Folders created:      {total_created}")
    print(f"  Folders existed:      {total_skipped}")
    print(f"  Errors:               {total_errors}")
    print("=" * 80)
    
    if total_errors > 0 and total_created == 0 and total_skipped == 0:
        raise RuntimeError(
            f"All folder creation attempts failed ({total_errors} errors). "
            f"Check lakehouse permissions and OneLake connectivity."
        )
    
    return total_created, total_skipped, total_errors

print("✓ create_lakehouse_folders() defined")


---

## Main Orchestration Function

The main function orchestrates the complete deployment workflow.

In [ ]:
def start_lakehouses_and_tables_deployment() -> None:
    """
    Main execution function that orchestrates the complete deployment workflow.
    
    Steps:
        1. Display configuration summary
        2. Get workspace context
        3. Create lakehouses
        4. Discover and create tables
        5. Create lakehouse folders
        6. Display completion summary
    """
    print("\n" + "=" * 80)
    print("LAKEHOUSES AND TABLES DEPLOYER")
    print("=" * 80)
    
    # Display configuration summary
    print("\nℹ️  Configuration Summary:")
    print(f"  Workspace: {WORKSPACE_NAME}")
    print(f"\nSource Configuration:")
    print(f"  Version: {ARTIFACT_VERSION}")
    print(f"  Tables Source Path: {TABLES_SOURCE_PATH}")
    print(f"\nLayer Mappings: {len(FOLDER_TO_LAKEHOUSE_MAP)} configured")
    for layer, lakehouses in FOLDER_TO_LAKEHOUSE_MAP.items():
        if isinstance(lakehouses, list):
            lakehouse_str = ", ".join(lakehouses)
            print(f"    • {layer} → {lakehouse_str}")
        else:
            print(f"    • {layer} → {lakehouses}")
    print(f"\nLakehouses to Create: {len(LAKEHOUSES_TO_CREATE)}")
    for lh in LAKEHOUSES_TO_CREATE:
        print(f"    • {lh}")
    print("=" * 80 + "\n")
    
    # Get workspace context (use WORKSPACE_ID from config)
    print(f"ℹ️  Using Workspace ID: {WORKSPACE_ID}\n")
    
    # Step 1: Create lakehouses
    created_lakehouses, existing_lakehouses = create_lakehouses(
        lakehouses_to_create=LAKEHOUSES_TO_CREATE,
        workspace_id=WORKSPACE_ID
    )
    
    # Step 2: Discover and create tables (with manifest filtering enabled)
    created_tables, skipped_tables, filtered_tables = discover_and_create_tables(
        source_base_path=TABLES_SOURCE_PATH,
        folder_to_lakehouse_map=FOLDER_TO_LAKEHOUSE_MAP,
        use_manifest_filter=True  # Enable manifest-based filtering
    )

    # Step 3: Create lakehouse folders (from manifest)
    folder_created = 0
    folder_skipped = 0
    folder_errors = 0
    
    folder_created, folder_skipped, folder_errors = create_lakehouse_folders(
        lakehouses_to_create=LAKEHOUSES_TO_CREATE,
        workspace_id=WORKSPACE_ID,
        endpoint_uri=ENDPOINT_URI
    )
    
    # Display completion summary
    print("\n" + "=" * 80)
    print("✅ DEPLOYMENT COMPLETE")
    print("=" * 80)
    print(f"\nLakehouses:")
    print(f"  • Created: {len(created_lakehouses)}")
    if created_lakehouses:
        for lh in created_lakehouses:
            print(f"    ✓ {lh}")
    print(f"  • Already existed: {len(existing_lakehouses)}")
    
    print(f"\nTables:")
    print(f"  • Created: {len(created_tables)}")
    if len(created_tables) > 0 and len(created_tables) <= 10:
        for table in created_tables:
            print(f"    ✓ {table}")
    elif len(created_tables) > 10:
        print(f"    (Showing first 10 of {len(created_tables)})")
        for table in created_tables[:10]:
            print(f"    ✓ {table}")
        print(f"    ... and {len(created_tables) - 10} more")
    
    print(f"\n  • Skipped (already existed): {len(skipped_tables)}")
    if len(skipped_tables) > 0 and len(skipped_tables) <= 10:
        for table in skipped_tables:
            print(f"    • {table}")
    elif len(skipped_tables) > 10:
        print(f"    (Showing first 10 of {len(skipped_tables)})")
        for table in skipped_tables[:10]:
            print(f"    • {table}")
        print(f"    ... and {len(skipped_tables) - 10} more")
    
    print(f"\n  • Filtered (not in manifest): {len(filtered_tables)}")
    if len(filtered_tables) > 0 and len(filtered_tables) <= 10:
        for table in filtered_tables:
            print(f"    ⏭️  {table}")
    elif len(filtered_tables) > 10:
        print(f"    (Showing first 10 of {len(filtered_tables)})")
        for table in filtered_tables[:10]:
            print(f"    ⏭️  {table}")
        print(f"    ... and {len(filtered_tables) - 10} more")
    
    print(f"\nFolders:")
    print(f"  • Created: {folder_created}")
    print(f"  • Already existed: {folder_skipped}")
    if folder_errors > 0:
        print(f"  • Errors: {folder_errors}")

    print(f"\n💡 Manifest-Based Filtering:")
    print(f"   Only tables defined in LakehouseHydrationManifest.json were deployed.")
    print(f"   To deploy all tables, set use_manifest_filter=False in the deployment call.")
    
    print(f"\nNext Steps:")
    print(f"  1. Verify lakehouses are created in the workspace")
    print(f"  2. Check tables are accessible and contain expected data")
    print(f"  3. Run data quality checks on created tables")
    print(f"  4. Review filtered tables if any were unexpected")
    print("=" * 80)

print("✓ start_lakehouses_and_tables_deployment() defined")

---

## Execute Deployment

Run the main function to execute the complete deployment workflow.

**⚠️ Important:** Review configuration in above Cells before running this cell.

In [ ]:
# Execute the deployment
start_lakehouses_and_tables_deployment()